## 1. Import thư viện và cấu hình

Notebook chạy trong working folder local chứa `TinHieuHuanLuyen` và `TinHieuKiemThu`. Nếu chạy từ thư mục khác, đặt `BT2_PROJECT_DIR` trỏ tới working folder.

In [ ]:
from pathlib import Path
import os
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.io.wavfile import WavFileWarning

# -------------------- Cấu hình bài toán --------------------
FRAME_MS_CANDIDATES = (20, 25, 30)
HOP_MS = 10
F0_MIN = 70.0
F0_MAX = 400.0
EPS = 1e-12
HIST_BINS = 20
HIST_SMOOTH_WINDOW = 5
HIST_W = 1.0

# Chỉ dùng để xử lý trường hợp số học suy biến: frame gần như toàn số 0.
# Đây KHÔNG phải threshold V/U học từ dữ liệu.
NEAR_ZERO_MEAN_ABS = 1e-8

# -------------------- Đường dẫn --------------------
PROJECT_DIR = Path(os.environ.get('BT2_PROJECT_DIR', Path.cwd())).expanduser().resolve()
TRAIN_DIR = PROJECT_DIR / 'TinHieuHuanLuyen'
TEST_DIR = PROJECT_DIR / 'TinHieuKiemThu'

print('PROJECT_DIR:', PROJECT_DIR)
print('TRAIN_DIR tồn tại:', TRAIN_DIR.exists())
print('TEST_DIR tồn tại :', TEST_DIR.exists())

## 2. Hàm đọc WAV, LAB và chia khung

Định dạng `.lab`:

- Các dòng đầu: `left_time  right_time  label`, với `label ∈ {sil, v, uv}`.
- Hai dòng cuối: `F0mean` và `F0std` (Hz).

Frame được gán ground-truth theo **thời điểm tâm frame**.

In [ ]:
def load_wav(path):
    """Đọc WAV, chuyển mono và chuẩn hóa biên độ về float64 gần [-1, 1]."""
    # Một số WAV có metadata chunk không chuẩn; bỏ warning vì không ảnh hưởng PCM data.
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', WavFileWarning)
        fs, signal = wavfile.read(path)

    if signal.ndim > 1:
        signal = signal.mean(axis=1)

    if np.issubdtype(signal.dtype, np.integer):
        info = np.iinfo(signal.dtype)
        scale = max(abs(info.min), info.max)
        signal = signal.astype(np.float64) / scale
    else:
        signal = signal.astype(np.float64)

    return fs, signal


def read_lab(path):
    """Đọc segment ground-truth và F0mean/F0std từ file .lab."""
    segments = []
    f0_mean = None
    f0_std = None

    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            parts = raw_line.strip().split()
            if not parts:
                continue

            key = parts[0]
            if key == 'F0mean':
                f0_mean = float(parts[1])
            elif key == 'F0std':
                f0_std = float(parts[1])
            else:
                start, end = float(parts[0]), float(parts[1])
                label = parts[2].lower()
                segments.append({'start': start, 'end': end, 'label': label})

    return segments, f0_mean, f0_std


def framing(signal, fs, frame_ms, hop_ms=HOP_MS):
    """
    Chia tín hiệu thành các frame bằng sliding_window_view.
    Trả về frames và thời điểm tâm của từng frame.
    """
    frame_len = int(round(fs * frame_ms / 1000.0))
    hop_len = int(round(fs * hop_ms / 1000.0))

    if len(signal) < frame_len:
        return np.empty((0, frame_len)), np.empty(0)

    windows = np.lib.stride_tricks.sliding_window_view(signal, frame_len)
    frames = windows[::hop_len]

    starts = np.arange(len(frames)) * hop_len
    center_times = (starts + frame_len / 2.0) / fs
    return frames, center_times


def label_at_time(time_sec, segments):
    """Lấy ground-truth label tại một thời điểm."""
    for seg in segments:
        if seg['start'] <= time_sec < seg['end']:
            return seg['label']
    return None


def pair_wav_lab(folder):
    """Ghép các cặp .wav / .lab theo cùng stem."""
    pairs = []
    for wav_path in sorted(folder.glob('*.wav')):
        lab_path = wav_path.with_suffix('.lab')
        if lab_path.exists():
            pairs.append((wav_path, lab_path))
    return pairs


train_pairs = pair_wav_lab(TRAIN_DIR)
test_pairs = pair_wav_lab(TEST_DIR)

print('Training pairs:', [p[0].name for p in train_pairs])
print('Testing pairs :', [p[0].name for p in test_pairs])

## 3. Normalized AMDF và tìm lowest local dip

AMDF cơ bản tại lag `k`:

\[
D(k)=\frac{1}{N-k}\sum_{n=0}^{N-k-1}|x[n]-x[n+k]|
\]

Để so sánh giữa các frame có mức biên độ khác nhau, notebook dùng normalized AMDF theo phần tín hiệu thực sự chồng nhau:

\[
D_N(k)=\frac{\operatorname{mean}|x[n]-x[n+k]|}
{\operatorname{mean}|x[n]|+\operatorname{mean}|x[n+k]|+\epsilon}
\]

- Tín hiệu **voiced**: khi `lag ≈ T0`, hai đoạn gần giống nhau → `D_N(k)` có **đáy thấp**.
- Tín hiệu **unvoiced**: không tuần hoàn → đáy thường cao hơn.
- Chỉ tìm đáy trong vùng lag tương ứng `70–400 Hz`.
- **Độ cao/độ sâu của đáy** dùng để V/U classification; **vị trí đáy** dùng để tính F0.

In [ ]:
def lag_search_range(fs, frame_len, f0_min=F0_MIN, f0_max=F0_MAX):
    """Tính miền lag hợp lệ từ dải F0."""
    lag_min = max(1, int(np.floor(fs / f0_max)))
    lag_max = min(frame_len - 1, int(np.ceil(fs / f0_min)))
    if lag_min > lag_max:
        raise ValueError('Frame quá ngắn so với miền lag cần tìm.')
    return lag_min, lag_max


def normalized_amdf(frame, fs, f0_min=F0_MIN, f0_max=F0_MAX):
    """
    Tính normalized AMDF trong miền lag hợp lệ.

    Vector hóa bằng NumPy theo cả trục lag và sample; chỉ tạo ma trận nhỏ
    tương ứng các lag 70–400 Hz, không tính các lag không cần thiết.
    """
    x = np.asarray(frame, dtype=np.float64)
    x = x - np.mean(x)  # loại DC nhỏ nếu có

    lag_min, lag_max = lag_search_range(fs, len(x), f0_min, f0_max)
    lags = np.arange(lag_min, lag_max + 1, dtype=int)

    # Frame gần như toàn số 0: AMDF chuẩn hóa bị suy biến 0/0.
    # Gán giá trị 1.0 để biểu diễn "không có bằng chứng tuần hoàn hữu ích".
    if np.mean(np.abs(x)) < NEAR_ZERO_MEAN_ABS:
        return lags, np.ones_like(lags, dtype=np.float64)

    max_overlap = len(x) - lag_min
    n = np.arange(max_overlap)[None, :]
    lag_matrix = lags[:, None]

    valid = n < (len(x) - lag_matrix)
    idx2 = np.minimum(n + lag_matrix, len(x) - 1)

    x1 = np.broadcast_to(x[:max_overlap][None, :], idx2.shape)
    x2 = x[idx2]

    count = valid.sum(axis=1)
    diff_mean = (np.abs(x1 - x2) * valid).sum(axis=1) / count
    x1_mean_abs = (np.abs(x1) * valid).sum(axis=1) / count
    x2_mean_abs = (np.abs(x2) * valid).sum(axis=1) / count

    namdf = diff_mean / (x1_mean_abs + x2_mean_abs + EPS)
    return lags, namdf


def deepest_local_dip(frame, fs, f0_min=F0_MIN, f0_max=F0_MAX):
    """
    Tìm lowest local dip của normalized AMDF.

    Returns
    -------
    dip_value : float
        Giá trị đáy AMDF dùng làm periodicity score.
    lag : int
        Lag nguyên tại đáy.
    refined_lag : float
        Lag tinh chỉnh bằng nội suy parabol để ước lượng F0 mượt hơn.
    lags, namdf : ndarray
        Dùng cho vẽ minh họa.
    """
    lags, namdf = normalized_amdf(frame, fs, f0_min, f0_max)

    if len(namdf) >= 3:
        local_mask = (namdf[1:-1] <= namdf[:-2]) & (namdf[1:-1] <= namdf[2:])
        local_idx = np.where(local_mask)[0] + 1
    else:
        local_idx = np.array([], dtype=int)

    # Nếu không có cực tiểu cục bộ nội bộ, fallback về minimum toàn cục.
    if len(local_idx) == 0:
        best_idx = int(np.argmin(namdf))
    else:
        best_idx = int(local_idx[np.argmin(namdf[local_idx])])

    dip_value = float(namdf[best_idx])
    lag = int(lags[best_idx])
    refined_lag = float(lag)

    # Nội suy parabol quanh đáy để giảm sai số lượng tử theo sample.
    if 0 < best_idx < len(namdf) - 1:
        y1, y2, y3 = namdf[best_idx - 1: best_idx + 2]
        denominator = y1 - 2.0 * y2 + y3
        if abs(denominator) > EPS:
            delta = 0.5 * (y1 - y3) / denominator
            if abs(delta) <= 1.0:
                refined_lag += float(delta)

    return dip_value, lag, refined_lag, lags, namdf


def f0_from_lag(fs, lag):
    return fs / lag if lag > 0 else np.nan

## 4. Thu thập lowest local dip từ tập huấn luyện

Với mỗi frame:

1. Xác định ground-truth `v` hoặc `uv` từ `.lab` tại thời điểm tâm frame.
2. Tính normalized AMDF.
3. Lấy **lowest local dip** trong miền lag 70–400 Hz.
4. Tách thành hai tập `V scores` và `U scores`.
5. Không dùng `sil` để tính `meanU/stdU`.

In [ ]:
def collect_training_scores(train_pairs, frame_ms, hop_ms=HOP_MS):
    voiced_scores = []
    unvoiced_scores = []

    for wav_path, lab_path in train_pairs:
        fs, signal = load_wav(wav_path)
        segments, _, _ = read_lab(lab_path)
        frames, center_times = framing(signal, fs, frame_ms, hop_ms)

        for frame, t in zip(frames, center_times):
            label = label_at_time(t, segments)
            if label not in {'v', 'uv'}:
                continue

            dip_value, _, _, _, _ = deepest_local_dip(frame, fs)
            if label == 'v':
                voiced_scores.append(dip_value)
            else:
                unvoiced_scores.append(dip_value)

    return np.asarray(voiced_scores), np.asarray(unvoiced_scores)


training_scores = {}
for frame_ms in FRAME_MS_CANDIDATES:
    v_scores, u_scores = collect_training_scores(train_pairs, frame_ms)
    training_scores[frame_ms] = {'v': v_scores, 'u': u_scores}



## 5. Ba phương pháp tìm threshold V/U trên training

Với AMDF, `dip < T` là voiced; `dip ≥ T` là unvoiced.

1. **Gaussian intersection:** fit hai phân bố chuẩn cho dip `v` và `uv`, chọn giao điểm gần giữa hai mean.
2. **Histogram hai mode M1/M2 [5]:** gộp dip `v/uv`, làm trơn histogram, tìm hai local maxima rồi tính `T=(W·M1+M2)/(W+1)` với `W=1`.
3. **Giao điểm histogram theo nhãn:** tạo histogram density riêng cho `v` và `uv` trên cùng các bin, làm trơn và tìm giao điểm gần giữa hai mean.

Cả ba ngưỡng dùng cùng frame training `v/uv`. Chọn theo **balanced accuracy V/UV**; accuracy thông thường là tie-break. `sil` không tham gia fit ngưỡng.


In [ ]:
def gaussian_threshold(v_scores, u_scores):
    """Chọn giao điểm Gaussian nằm giữa meanV và meanU, giống quy tắc của ACF."""
    mu_v, sv = float(v_scores.mean()), max(float(v_scores.std()), EPS)
    mu_u, su = float(u_scores.mean()), max(float(u_scores.std()), EPS)

    # Giải log pV(x) = log pU(x). Hệ số được viết cùng dạng với notebook ACF.
    a = 1.0 / su**2 - 1.0 / sv**2
    b = -2.0 * mu_u / su**2 + 2.0 * mu_v / sv**2
    c = (
        mu_u**2 / su**2
        - mu_v**2 / sv**2
        + 2.0 * np.log(su / sv)
    )

    if abs(a) < EPS:
        roots = [] if abs(b) < EPS else [-c / b]
    else:
        roots = np.roots([a, b, c])

    real_roots = [
        float(np.real(root))
        for root in roots
        if abs(np.imag(root)) < 1e-9
    ]

    lo, hi = sorted((mu_v, mu_u))
    between = [root for root in real_roots if lo <= root <= hi]
    midpoint = 0.5 * (mu_v + mu_u)

    if between:
        return min(between, key=lambda root: abs(root - midpoint))
    if real_roots:
        return min(real_roots, key=lambda root: abs(root - midpoint))
    return midpoint

def histogram_threshold(v_scores, u_scores, bins=80, smooth_width=7):
    """
    Histogram smoothing cho hai lớp V/UV rồi tìm điểm giao density.
    Trả thêm dữ liệu histogram để vẽ.
    """
    all_scores = np.concatenate([v_scores, u_scores])
    x_min, x_max = float(all_scores.min()), float(all_scores.max())

    hist_v, edges = np.histogram(v_scores, bins=bins, range=(x_min, x_max), density=True)
    hist_u, _ = np.histogram(u_scores, bins=bins, range=(x_min, x_max), density=True)

    kernel = np.ones(smooth_width, dtype=float) / smooth_width
    smooth_v = np.convolve(hist_v, kernel, mode='same')
    smooth_u = np.convolve(hist_u, kernel, mode='same')
    centers = 0.5 * (edges[:-1] + edges[1:])

    lo, hi = sorted((float(v_scores.mean()), float(u_scores.mean())))
    candidate_idx = np.where((centers >= lo) & (centers <= hi))[0]
    if len(candidate_idx) == 0:
        threshold = 0.5 * (lo + hi)
        return threshold, centers, smooth_v, smooth_u

    diff = smooth_v - smooth_u
    crossings = []
    for idx in candidate_idx[:-1]:
        if diff[idx] == 0 or diff[idx] * diff[idx + 1] < 0:
            best = idx if abs(diff[idx]) <= abs(diff[idx + 1]) else idx + 1
            crossings.append(best)

    if crossings:
        midpoint = 0.5 * (lo + hi)
        best_idx = min(crossings, key=lambda i: abs(centers[i] - midpoint))
    else:
        best_idx = candidate_idx[np.argmin(np.abs(diff[candidate_idx]))]

    threshold = float(centers[best_idx])
    return threshold, centers, smooth_v, smooth_u


def two_mode_histogram_threshold(scores, bins=HIST_BINS,
                                 smooth_window=HIST_SMOOTH_WINDOW, W=HIST_W):
    """Histogram gộp -> smoothing -> hai local maxima -> T=(W*M1+M2)/(W+1)."""
    scores = np.asarray(scores, dtype=float)
    scores = scores[np.isfinite(scores)]
    if len(scores) < 2:
        raise ValueError('Không đủ score để tính histogram hai mode.')
    hist, edges = np.histogram(scores, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    smooth_window = int(max(1, smooth_window))
    if smooth_window % 2 == 0:
        smooth_window += 1
    kernel = np.ones(smooth_window, dtype=float) / smooth_window
    smooth_hist = np.convolve(hist.astype(float), kernel, mode='same')
    local_idx = np.where(
        (smooth_hist[1:-1] >= smooth_hist[:-2]) &
        (smooth_hist[1:-1] > smooth_hist[2:])
    )[0] + 1
    ranked = list(local_idx[np.argsort(smooth_hist[local_idx])[::-1]])
    if len(ranked) < 2:
        ranked = list(np.argsort(smooth_hist)[::-1])
    chosen = []
    for idx in ranked:
        if all(abs(int(idx) - int(j)) >= 2 for j in chosen):
            chosen.append(int(idx))
        if len(chosen) == 2:
            break
    if len(chosen) < 2:
        chosen = [int(np.argmin(centers)), int(np.argmax(centers))]
    m1, m2 = sorted([float(centers[chosen[0]]), float(centers[chosen[1]])])
    return {
        'threshold': float((W * m1 + m2) / (W + 1.0)),
        'M1': m1, 'M2': m2,
        'hist': hist, 'smooth_hist': smooth_hist,
        'centers': centers, 'edges': edges,
        'peak_indices': np.array(chosen, dtype=int),
    }


def vu_metrics(v_scores, u_scores, threshold):
    """Đánh giá binary V/UV với quy tắc dip < T => V."""
    pred_v_for_v = v_scores < threshold
    pred_v_for_u = u_scores < threshold

    tp = int(pred_v_for_v.sum())
    fn = len(v_scores) - tp
    fp = int(pred_v_for_u.sum())
    tn = len(u_scores) - fp

    accuracy = (tp + tn) / (len(v_scores) + len(u_scores))
    recall_v = tp / len(v_scores)
    recall_u = tn / len(u_scores)
    balanced_accuracy = 0.5 * (recall_v + recall_u)

    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'recall_v': recall_v,
        'recall_u': recall_u,
    }


def gaussian_pdf(x, mean, std):
    std = max(float(std), 1e-8)
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2.0 * np.pi))

In [ ]:
training_rows = []
threshold_candidates = {}

for frame_ms in FRAME_MS_CANDIDATES:
    v_scores = training_scores[frame_ms]['v']
    u_scores = training_scores[frame_ms]['u']
    t_gauss = gaussian_threshold(v_scores, u_scores)
    mode_info = two_mode_histogram_threshold(np.concatenate([v_scores, u_scores]))
    t_modes = mode_info['threshold']
    t_classes, hist_x, hist_v, hist_u = histogram_threshold(v_scores, u_scores)
    methods = {
        'Gaussian': t_gauss,
        'Histogram M1/M2': t_modes,
        'Histogram V/UV intersection': t_classes,
    }
    method_metrics = {
        name: vu_metrics(v_scores, u_scores, threshold)
        for name, threshold in methods.items()
    }
    chosen = max(methods, key=lambda name: (
        method_metrics[name]['balanced_accuracy'], method_metrics[name]['accuracy']
    ))
    threshold_candidates[frame_ms] = {
        'Gaussian': t_gauss, 'Modes': t_modes, 'Classes': t_classes,
        'chosen_method': chosen, 'chosen_threshold': methods[chosen],
        'chosen_metrics': method_metrics[chosen], 'method_metrics': method_metrics,
        'mode_info': mode_info,
        'hist_x': hist_x, 'hist_v': hist_v, 'hist_u': hist_u,
    }
    for method, threshold in methods.items():
        metrics = method_metrics[method]
        training_rows.append({
            'Frame (ms)': frame_ms, 'Method': method,
            'Threshold': threshold,
            'meanV': float(v_scores.mean()), 'stdV': float(v_scores.std()),
            'meanU': float(u_scores.mean()), 'stdU': float(u_scores.std()),
            'Accuracy V/UV': metrics['accuracy'],
            'Balanced accuracy V/UV': metrics['balanced_accuracy'],
            'Selected': 'Yes' if method == chosen else '',
        })

training_methods_df = pd.DataFrame(training_rows)
training_df = training_methods_df  # dữ liệu số giữ nguyên cho phân tích tiếp
training_display = training_methods_df.copy()
for column in ['Threshold', 'meanV', 'stdV', 'meanU', 'stdU']:
    training_display[column] = training_display[column].map(lambda value: f'{value:.4f}')
for column in ['Accuracy V/UV', 'Balanced accuracy V/UV']:
    training_display[column] = training_display[column].map(lambda value: f'{value:.2%}')
print('Threshold methods on training V/UV frames:')
display(training_display)


## 6. Ba loại đồ thị phân bố và threshold

Cell kế tiếp xuất ba figure riêng: Gaussian intersection, histogram hai mode M1/M2 và giao điểm hai histogram theo nhãn V/UV. Mỗi figure gồm ba panel cho 20/25/30 ms và chỉ vẽ threshold của chính phương pháp đó.


In [ ]:
fig_gaussian, axes_gaussian = plt.subplots(1, len(FRAME_MS_CANDIDATES), figsize=(5.5 * len(FRAME_MS_CANDIDATES), 4.5))
fig_modes, axes_modes = plt.subplots(1, len(FRAME_MS_CANDIDATES), figsize=(5.5 * len(FRAME_MS_CANDIDATES), 4.5))
fig_classes, axes_classes = plt.subplots(1, len(FRAME_MS_CANDIDATES), figsize=(5.5 * len(FRAME_MS_CANDIDATES), 4.5))
axes = np.array([axes_gaussian, axes_modes, axes_classes])

for col, frame_ms in enumerate(FRAME_MS_CANDIDATES):
    v_scores = training_scores[frame_ms]['v']
    u_scores = training_scores[frame_ms]['u']
    info = threshold_candidates[frame_ms]
    mode = info['mode_info']
    x = np.linspace(min(v_scores.min(), u_scores.min()),
                    max(v_scores.max(), u_scores.max()), 500)

    ax = axes[0, col]
    ax.hist(v_scores, bins=35, density=True, alpha=0.30, label='Voiced dips')
    ax.hist(u_scores, bins=35, density=True, alpha=0.30, label='Unvoiced dips')
    ax.plot(x, gaussian_pdf(x, v_scores.mean(), v_scores.std()), label='Gaussian V')
    ax.plot(x, gaussian_pdf(x, u_scores.mean(), u_scores.std()), label='Gaussian UV')
    ax.axvline(info['Gaussian'], linestyle='--', linewidth=2,
               label=f"T={info['Gaussian']:.3f}")
    ax.set_title(f"Gaussian · {frame_ms} ms · BalAcc {info['method_metrics']['Gaussian']['balanced_accuracy']:.2%}")
    ax.set_xlabel('Lowest local dip của AMDF')
    ax.set_ylabel('Density')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

    ax = axes[1, col]
    ax.bar(mode['centers'], mode['hist'], width=np.diff(mode['edges']),
           alpha=0.35, align='center', label='Histogram gộp')
    ax.plot(mode['centers'], mode['smooth_hist'], linewidth=2,
            label='Smoothed histogram')
    ax.axvline(mode['M1'], linestyle='--', label=f"M1={mode['M1']:.3f}")
    ax.axvline(mode['M2'], linestyle='--', label=f"M2={mode['M2']:.3f}")
    ax.axvline(info['Modes'], linestyle=':', linewidth=2,
               label=f"T={info['Modes']:.3f}")
    ax.set_title(f"Histogram M1/M2 · {frame_ms} ms · BalAcc {info['method_metrics']['Histogram M1/M2']['balanced_accuracy']:.2%}")
    ax.set_xlabel('Lowest local dip của AMDF')
    ax.set_ylabel('Frame count')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

    ax = axes[2, col]
    ax.plot(info['hist_x'], info['hist_v'], linewidth=2,
            label='Smoothed histogram V')
    ax.plot(info['hist_x'], info['hist_u'], linewidth=2,
            label='Smoothed histogram UV')
    ax.axvline(info['Classes'], linestyle=':', linewidth=2,
               label=f"T={info['Classes']:.3f}")
    ax.set_title(f"Histogram V/UV intersection · {frame_ms} ms · BalAcc {info['method_metrics']['Histogram V/UV intersection']['balanced_accuracy']:.2%}")
    ax.set_xlabel('Lowest local dip của AMDF')
    ax.set_ylabel('Density')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

for fig in (fig_gaussian, fig_modes, fig_classes):
    fig.tight_layout()
plt.show()


## 7. Chọn frame length và threshold cuối cùng

Với mỗi frame length, chọn trong ba phương pháp theo balanced accuracy V/UV trên training; accuracy V/UV là tie-break. Sau đó chọn 20, 25 hoặc 30 ms theo cùng tiêu chí. Test không dùng để chọn tham số.


In [ ]:
def frame_selection_key(frame_ms):
    metrics = threshold_candidates[frame_ms]['chosen_metrics']
    return (metrics['balanced_accuracy'], metrics['accuracy'])

FINAL_FRAME_MS = max(FRAME_MS_CANDIDATES, key=frame_selection_key)
FINAL_METHOD = threshold_candidates[FINAL_FRAME_MS]['chosen_method']
FINAL_THRESHOLD = threshold_candidates[FINAL_FRAME_MS]['chosen_threshold']
FINAL_TRAIN_METRICS = threshold_candidates[FINAL_FRAME_MS]['chosen_metrics']
FINAL_THRESHOLD_METHOD = FINAL_METHOD

selection_rows = []
for frame_ms in FRAME_MS_CANDIDATES:
    info = threshold_candidates[frame_ms]
    metrics = info['chosen_metrics']
    selection_rows.append({
        'Frame (ms)': int(frame_ms),
        'Chosen method': info['chosen_method'],
        'Threshold': float(info['chosen_threshold']),
        'Accuracy V/UV': float(metrics['accuracy']),
        'Balanced accuracy V/UV': float(metrics['balanced_accuracy']),
    })
selection_df = pd.DataFrame(selection_rows)
selection_display = selection_df.copy()
selection_display['Threshold'] = selection_display['Threshold'].map(lambda value: f'{value:.4f}')
for column in ['Accuracy V/UV', 'Balanced accuracy V/UV']:
    selection_display[column] = selection_display[column].map(lambda value: f'{value:.2%}')
print('Chosen method for each frame length:')
display(selection_display)
print(f'Final configuration: frame {FINAL_FRAME_MS} ms; shift {HOP_MS} ms; '
      f'F0 {F0_MIN:.0f}–{F0_MAX:.0f} Hz; threshold {FINAL_THRESHOLD:.4f} '
      f'({FINAL_THRESHOLD_METHOD}); training balanced accuracy '
      f'{FINAL_TRAIN_METRICS["balanced_accuracy"]:.2%}')


## 8. Minh họa AMDF: 1 voiced frame và 1 unvoiced frame

Để minh họa rõ ảnh hưởng của độ dài khung và so sánh với ACF, notebook chọn cùng một file/segment `.lab` theo thứ tự tên file: midpoint của segment `v` và `uv` đầu tiên dài ít nhất 50 ms. Quy tắc chọn độc lập với score của thuật toán.

Sau đó lấy **cùng thời điểm** và vẽ AMDF ở `20 ms`, `25 ms` và `30 ms`.

- Với voiced: nếu `dip < threshold`, lấy vị trí đáy làm `T0` và tính `F0 = fs / lag`.
- Với unvoiced: nếu `dip >= threshold`, kết luận `F0 không xác định`.
Sau hai figure demo, notebook in cùng bảng so sánh `Example/File/Center/GT/Frame/Score/Threshold/Predicted/Best lag/F0` như ACF.


In [ ]:
def extract_centered_frame(signal, fs, center_time, frame_ms):
    frame_len = int(round(fs * frame_ms / 1000.0))
    center = int(round(center_time * fs))
    start = center - frame_len // 2
    start = max(0, min(start, len(signal) - frame_len))
    return signal[start:start + frame_len]


def find_representative_segment(train_pairs, target_label):
    """Chọn cùng file/segment theo thứ tự LAB như demo ACF, độc lập với score."""
    for wav_path, lab_path in train_pairs:
        segments, _, _ = read_lab(lab_path)
        for seg in segments:
            if seg['label'] == target_label and seg['end'] - seg['start'] >= 0.05:
                return {
                    'wav_path': wav_path,
                    'lab_path': lab_path,
                    'time': 0.5 * (seg['start'] + seg['end']),
                }
    raise RuntimeError(f'Không tìm thấy segment {target_label} đủ dài để minh họa.')


def plot_amdf_demo(candidate, label_name):
    fs, signal = load_wav(candidate['wav_path'])
    center_time = candidate['time']

    rows = []
    fig, axes = plt.subplots(1, len(FRAME_MS_CANDIDATES),
                             figsize=(5 * len(FRAME_MS_CANDIDATES), 4.8))

    for ax, frame_ms in zip(axes, FRAME_MS_CANDIDATES):
        frame = extract_centered_frame(signal, fs, center_time, frame_ms)
        dip_value, lag, refined_lag, lags, namdf = deepest_local_dip(frame, fs)
        threshold = threshold_candidates[frame_ms]['chosen_threshold']

        predicted_voiced = dip_value < threshold
        f0 = f0_from_lag(fs, refined_lag) if predicted_voiced else np.nan

        ax.plot(lags, namdf, label='Normalized AMDF')
        ax.scatter([lag], [dip_value], s=55, zorder=3,
                   label=f'Deepest local dip={dip_value:.3f}')
        ax.axhline(threshold, linestyle='--', label=f'Threshold={threshold:.3f}')

        result_text = f'F0 ≈ {f0:.2f} Hz' if np.isfinite(f0) else 'F0 không xác định'
        ax.set_title(f'{label_name} — {frame_ms} ms\nlag={refined_lag:.2f} samples | {result_text}')
        ax.set_xlabel('Lag (samples)')
        ax.set_ylabel('Normalized AMDF')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

        rows.append({
            'example': label_name, 'file': candidate['wav_path'].name,
            'center_s': center_time, 'ground_truth': 'v' if label_name == 'Voiced' else 'uv',
            'frame_ms': frame_ms, 'score': dip_value, 'threshold': threshold,
            'predicted': 'v' if predicted_voiced else 'uv',
            'best_lag_samples': refined_lag, 'F0_Hz': f0,
        })

    plt.suptitle(
        f"{candidate['wav_path'].name} | center={center_time:.3f}s | Ground-truth={label_name}"
    )
    plt.tight_layout()
    plt.show()
    return rows


voiced_demo = find_representative_segment(train_pairs, 'v')
unvoiced_demo = find_representative_segment(train_pairs, 'uv')

comparison_rows = (
    plot_amdf_demo(voiced_demo, 'Voiced')
    + plot_amdf_demo(unvoiced_demo, 'Unvoiced')
)
comparison_df = pd.DataFrame(comparison_rows)
comparison_display = comparison_df.rename(columns={
    'example': 'Example', 'file': 'File', 'center_s': 'Center (s)',
    'ground_truth': 'GT', 'frame_ms': 'Frame (ms)',
    'score': 'Score', 'threshold': 'Threshold',
    'predicted': 'Predicted', 'best_lag_samples': 'Best lag (samples)',
    'F0_Hz': 'F0 (Hz)',
})[[
    'Example', 'File', 'Center (s)', 'GT', 'Frame (ms)',
    'Score', 'Threshold', 'Predicted', 'Best lag (samples)', 'F0 (Hz)'
]]
print('Same-center V/UV examples across frame lengths:')
display(comparison_display.round(4))


### Vì sao AMDF giúp tìm F0?

Khi tín hiệu hữu thanh gần tuần hoàn, dịch tín hiệu đúng một chu kỳ `T0` làm hai đoạn gần trùng nhau nên trung bình `|x[n] - x[n+k]|` giảm mạnh. Vì vậy AMDF xuất hiện đáy tại lag gần `T0` (và có thể tại các bội của `T0`). Sau khi giới hạn lag về dải 70–400 Hz, lowest local dip được dùng để ước lượng:

\[
T_0 \approx \frac{lag}{f_s}, \qquad F_0 \approx \frac{f_s}{lag}
\]

Với unvoiced, cấu trúc tín hiệu không tuần hoàn nên dịch theo lag không tạo được sự triệt tiêu ổn định; lowest dip thường cao hơn threshold và F0 không được gán.

## 9. Hàm xử lý một file kiểm thử

Quy trình từng frame:

1. Tính lowest local dip của normalized AMDF.
2. `dip < FINAL_THRESHOLD` → predicted Voiced.
3. Voiced → `F0 = fs / refined_lag`; ngược lại `F0 = NaN`.
4. Báo cáo riêng **Accuracy V/UV** (chỉ `v`, `uv`) và **Accuracy V/UV/SIL** (thêm `sil` là không voiced). Nhãn lấy tại tâm frame.

In [ ]:
def process_file_amdf(wav_path, lab_path, frame_ms=FINAL_FRAME_MS,
                      hop_ms=HOP_MS, threshold=FINAL_THRESHOLD):
    fs, signal = load_wav(wav_path)
    segments, gt_f0_mean, gt_f0_std = read_lab(lab_path)
    frames, center_times = framing(signal, fs, frame_ms, hop_ms)

    scores = np.empty(len(frames), dtype=float)
    predicted_voiced = np.zeros(len(frames), dtype=bool)
    f0_contour = np.full(len(frames), np.nan, dtype=float)
    gt_labels = np.empty(len(frames), dtype=object)

    for i, (frame, t) in enumerate(zip(frames, center_times)):
        gt_labels[i] = label_at_time(t, segments)
        dip_value, _, refined_lag, _, _ = deepest_local_dip(frame, fs)
        scores[i] = dip_value

        if dip_value < threshold:
            predicted_voiced[i] = True
            f0_contour[i] = f0_from_lag(fs, refined_lag)

    # Hai phạm vi đánh giá trên cùng dự đoán; SIL là non-voiced ở phạm vi toàn bộ.
    mask_vu = np.isin(gt_labels, ['v', 'uv'])
    mask_all = np.isin(gt_labels, ['v', 'uv', 'sil'])
    gt_voiced = gt_labels == 'v'
    n_vu, n_all = int(mask_vu.sum()), int(mask_all.sum())
    correct_vu = int(np.sum(gt_voiced[mask_vu] == predicted_voiced[mask_vu]))
    correct_all = int(np.sum(gt_voiced[mask_all] == predicted_voiced[mask_all]))
    accuracy_vu = correct_vu / n_vu if n_vu else np.nan
    accuracy_all = correct_all / n_all if n_all else np.nan

    valid_f0 = f0_contour[np.isfinite(f0_contour)]
    est_f0_mean = float(np.mean(valid_f0)) if len(valid_f0) else np.nan
    est_f0_std = float(np.std(valid_f0, ddof=0)) if len(valid_f0) else np.nan

    return {
        'fs': fs,
        'signal': signal,
        'segments': segments,
        'gt_f0_mean': gt_f0_mean,
        'gt_f0_std': gt_f0_std,
        'center_times': center_times,
        'scores': scores,
        'predicted_voiced': predicted_voiced,
        'f0_contour': f0_contour,
        'est_f0_mean': est_f0_mean,
        'est_f0_std': est_f0_std,
        'n_vu': n_vu,
        'correct_vu': correct_vu,
        'accuracy_vu': accuracy_vu,
        'n_all': n_all,
        'correct_all': correct_all,
        'accuracy_all': accuracy_all,
        'sil_false_voiced': int(np.sum((gt_labels == 'sil') & predicted_voiced)),
    }


def plot_test_result(file_name, result):
    fs, signal = result['fs'], result['signal']
    segments = result['segments']
    times = result['center_times']
    f0 = result['f0_contour']
    gt_mean, gt_std = result['gt_f0_mean'], result['gt_f0_std']
    t_signal = np.arange(len(signal)) / fs

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(t_signal, signal, linewidth=0.8)
    first_gt = True
    for seg in segments:
        if seg['label'] == 'v':
            axes[0].axvspan(seg['start'], seg['end'], color='blue', alpha=0.12,
                            label='Ground-truth voiced' if first_gt else None)
            first_gt = False

    pred_voiced = result['predicted_voiced']
    hop_s = HOP_MS / 1000.0
    in_voiced = False
    first_pred = True
    for i, is_voiced in enumerate(pred_voiced):
        if is_voiced and not in_voiced:
            start_t = times[i] - hop_s / 2
            in_voiced = True
        elif not is_voiced and in_voiced:
            axes[0].axvspan(start_t, times[i - 1] + hop_s / 2,
                            color='green', alpha=0.25,
                            label='Predicted voiced' if first_pred else None)
            first_pred = False
            in_voiced = False
    if in_voiced:
        axes[0].axvspan(start_t, times[-1] + hop_s / 2,
                        color='green', alpha=0.25,
                        label='Predicted voiced' if first_pred else None)
    axes[0].set_title(f'Input waveform – {file_name}')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(alpha=0.25)
    axes[0].legend(loc='upper right')

    axes[1].plot(times, f0, '.-', markersize=3, linewidth=1,
                 label='Estimated F0 contour')
    if gt_mean is not None:
        axes[1].axhline(gt_mean, linestyle='--', linewidth=1.5,
                        label=f'LAB F0mean={gt_mean:.1f} Hz')
    if gt_mean is not None and gt_std is not None:
        axes[1].axhspan(gt_mean - gt_std, gt_mean + gt_std, alpha=0.10,
                        label=f'LAB F0mean ± F0std ({gt_std:.1f} Hz)')
    axes[1].set_ylim(0, F0_MAX + 30)
    axes[1].set_xlabel('Thời gian (s)')
    axes[1].set_ylabel('F0 (Hz)')
    axes[1].set_title(
        f"Output – F0mean={result['est_f0_mean']:.2f} Hz, "
        f"F0std={result['est_f0_std']:.2f} Hz, "
        f"Acc(V/UV)={result['accuracy_vu']:.2%}, "
        f"Acc(V/UV/SIL)={result['accuracy_all']:.2%}"
    )
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc='upper right')
    plt.tight_layout()
    plt.show()


## 10. Kiểm thử bốn file

Như ACF, cell kế tiếp tạo bốn figure, bảng F0 theo `TEST WAV` và bảng hai accuracy theo file. Cả hai notebook dùng cùng tên cột, đơn vị, số chữ số và định nghĩa metric; AMDF dùng dip thấp là voiced, ACF dùng peak cao là voiced.


In [ ]:
def macro_f1_vu(labels, predictions):
    """Macro F1 của hai lớp V và UV, không dùng SIL."""
    labels = np.asarray(labels)
    predictions = np.asarray(predictions, dtype=bool)
    mask = np.isin(labels, ['v', 'uv'])
    truth = labels[mask] == 'v'
    pred = predictions[mask]
    tp = int(np.sum(truth & pred))
    tn = int(np.sum(~truth & ~pred))
    fp = int(np.sum(~truth & pred))
    fn = int(np.sum(truth & ~pred))
    f1_v = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0
    f1_uv = 2 * tn / (2 * tn + fp + fn) if (2 * tn + fp + fn) else 0.0
    return 0.5 * (f1_v + f1_uv) if mask.any() else np.nan


test_summaries = []
for wav_path, lab_path in test_pairs:
    result = process_file_amdf(
        wav_path, lab_path,
        frame_ms=FINAL_FRAME_MS, hop_ms=HOP_MS, threshold=FINAL_THRESHOLD,
    )
    plot_test_result(wav_path.name, result)
    labels = np.asarray([
        label_at_time(t, result['segments']) for t in result['center_times']
    ], dtype=object)
    gt_mean, gt_std = result['gt_f0_mean'], result['gt_f0_std']
    est_mean, est_std = result['est_f0_mean'], result['est_f0_std']
    vu = np.isin(labels, ['v', 'uv'])
    truth = labels[vu] == 'v'
    pred = result['predicted_voiced'][vu]
    test_summaries.append({
        'TP': int(np.sum(truth & pred)), 'FN': int(np.sum(truth & ~pred)),
        'FP': int(np.sum(~truth & pred)), 'TN': int(np.sum(~truth & ~pred)),
        'file': wav_path.name,
        'F0mean_GT': gt_mean, 'F0mean_est': est_mean,
        'abs_error_mean': abs(est_mean - gt_mean) if np.isfinite(est_mean) and gt_mean is not None else np.nan,
        'F0std_GT': gt_std, 'F0std_est': est_std,
        'abs_error_std': abs(est_std - gt_std) if np.isfinite(est_std) and gt_std is not None else np.nan,
        'NumF0': int(np.isfinite(result['f0_contour']).sum()),
        'correct_vu': result['correct_vu'], 'n_vu': result['n_vu'],
        'accuracy_vu': result['accuracy_vu'],
        'correct_all': result['correct_all'], 'n_all': result['n_all'],
        'accuracy_all': result['accuracy_all'],
        'sil_false_voiced': result['sil_false_voiced'],
        'macro_f1_vu': macro_f1_vu(labels, result['predicted_voiced']),
    })

test_summary_df = pd.DataFrame(test_summaries)

# Mỗi file chỉ có một F0mean/F0std trong LAB, nên MAE ở đây là sai số tuyệt đối của file đó.
f0_report = test_summary_df[[
    'file', 'F0mean_GT', 'F0mean_est', 'abs_error_mean',
    'F0std_GT', 'F0std_est', 'abs_error_std', 'NumF0'
]].rename(columns={
    'file': 'TEST WAV',
    'F0mean_GT': 'LAB F0mean', 'F0mean_est': 'Pred F0mean',
    'abs_error_mean': 'F0mean MAE',
    'F0std_GT': 'LAB F0std', 'F0std_est': 'Pred F0std',
    'abs_error_std': 'F0std MAE'
})
print('Per-file F0 metrics (all F0 values in Hz; NumF0 is a frame count):')
display(f0_report.round(4))

classification_report = test_summary_df[[
    'file', 'correct_vu', 'n_vu', 'accuracy_vu',
    'correct_all', 'n_all', 'accuracy_all', 'sil_false_voiced'
]].rename(columns={
    'file': 'TEST WAV',
    'correct_vu': 'Correct V/UV', 'n_vu': 'N V/UV',
    'accuracy_vu': 'Accuracy V/UV',
    'correct_all': 'Correct V/UV/SIL', 'n_all': 'N V/UV/SIL',
    'accuracy_all': 'Accuracy V/UV/SIL',
    'sil_false_voiced': 'False voiced on SIL'
})
classification_display = classification_report.copy()
for column in ['Accuracy V/UV', 'Accuracy V/UV/SIL']:
    classification_display[column] = classification_display[column].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—'
    )
print('Per-file classification accuracy:')
display(classification_display)


## 11. Tổng kết tự động

MAE, RMSE, MAPE được tính trên bốn cặp thống kê F0 theo file; Macro F1 (V/UV) và hai macro accuracy là trung bình không trọng số qua các file. Các bảng có cùng cột và cùng định dạng với ACF.


In [ ]:
def aggregate_f0_metrics(gt, pred):
    """MAE/RMSE/MAPE trên các thống kê F0 theo file, bỏ giá trị thiếu; MAPE bỏ GT=0."""
    gt = np.asarray(gt, dtype=float)
    pred = np.asarray(pred, dtype=float)
    valid = np.isfinite(gt) & np.isfinite(pred)
    errors = pred[valid] - gt[valid]
    mae = float(np.mean(np.abs(errors))) if len(errors) else np.nan
    rmse = float(np.sqrt(np.mean(errors ** 2))) if len(errors) else np.nan
    valid_pct = valid & (gt != 0)
    mape = (float(np.mean(np.abs((pred[valid_pct] - gt[valid_pct]) / gt[valid_pct])))
            if valid_pct.any() else np.nan)
    return mae, rmse, mape


mean_mae, mean_rmse, mean_mape = aggregate_f0_metrics(
    test_summary_df['F0mean_GT'], test_summary_df['F0mean_est']
)
std_mae, std_rmse, std_mape = aggregate_f0_metrics(
    test_summary_df['F0std_GT'], test_summary_df['F0std_est']
)
f0_aggregate = pd.DataFrame([{
    'files': len(test_summary_df),
    'F0mean MAE': mean_mae, 'F0mean RMSE': mean_rmse,
    'F0mean MAPE': mean_mape,
    'F0std MAE': std_mae, 'F0std RMSE': std_rmse,
    'F0std MAPE': std_mape,
    'Macro F1 (V/UV)': test_summary_df['macro_f1_vu'].mean(),
}])
f0_aggregate_display = f0_aggregate.copy()
for column in ['F0mean MAE', 'F0mean RMSE', 'F0std MAE', 'F0std RMSE', 'Macro F1 (V/UV)']:
    f0_aggregate_display[column] = f0_aggregate_display[column].map(
        lambda value: f'{value:.4f}' if pd.notna(value) else '—'
    )
for column in ['F0mean MAPE', 'F0std MAPE']:
    f0_aggregate_display[column] = f0_aggregate_display[column].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—'
    )
print('Aggregate F0 metrics (MAE/RMSE in Hz; MAPE in %; Macro F1 averaged over files):')
display(f0_aggregate_display)

accuracy_aggregate = pd.DataFrame([{
    'files': len(test_summary_df),
    'Macro accuracy V/UV': test_summary_df['accuracy_vu'].mean(),
    'Macro accuracy V/UV/SIL': test_summary_df['accuracy_all'].mean(),
}])
accuracy_aggregate_display = accuracy_aggregate.copy()
for column in ['Macro accuracy V/UV', 'Macro accuracy V/UV/SIL']:
    accuracy_aggregate_display[column] = accuracy_aggregate_display[column].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—'
    )
print('Aggregate classification accuracy (unweighted mean over files):')
display(accuracy_aggregate_display)

print(f'AMDF configuration: frame {FINAL_FRAME_MS} ms; shift {HOP_MS} ms; '
      f'F0 {F0_MIN:.0f}–{F0_MAX:.0f} Hz; threshold {FINAL_THRESHOLD:.4f} ({FINAL_THRESHOLD_METHOD})')


In [ ]:
# Chẩn đoán V/UV: số khung và tỷ lệ theo nhãn thật trên tập test.
# Đây là mô tả cấu hình đã chọn từ training; không dùng hình này để chọn ngưỡng.
if 'Algorithm' in test_summary_df.columns:
    confusion_groups = list(test_summary_df.groupby('Algorithm', sort=False))
else:
    confusion_groups = [('AMDF', test_summary_df)]
fig, axes = plt.subplots(1, len(confusion_groups), figsize=(5.2 * len(confusion_groups), 4.4), squeeze=False, layout='constrained')
for ax, (name, group) in zip(axes[0], confusion_groups):
    cm = np.array([[group['TP'].sum(), group['FN'].sum()],
                   [group['FP'].sum(), group['TN'].sum()]], dtype=int)
    assert int(cm.sum()) == int(group['n_vu'].sum())
    assert int(cm[0, 0] + cm[1, 1]) == int(group['correct_vu'].sum())
    row_pct = 100 * cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(row_pct, cmap='Blues', vmin=0, vmax=100)
    for row in range(2):
        for col in range(2):
            color = 'white' if row_pct[row, col] > 55 else '#17202a'
            ax.text(col, row, f'{cm[row, col]}\n({row_pct[row, col]:.1f}%)',
                    ha='center', va='center', fontsize=12, color=color)
    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=['V', 'UV'], yticklabels=['V', 'UV'],
           xlabel='Dự đoán', ylabel='Nhãn LAB', title=f'{name} · {int(cm.sum())} khung V/UV')
fig.suptitle('Test V/UV: số khung và % theo nhãn thật')
plt.show()
plt.close(fig)


In [ ]:
# Metric theo file: tách tỷ lệ V/UV và sai số F0 để tránh trộn đơn vị.
# Chỉ minh họa kết quả TEST của cấu hình đã chọn trên training; không dò tham số từ biểu đồ.
plot_groups = (test_summary_df.groupby('Algorithm', sort=False)
               if 'Algorithm' in test_summary_df.columns
               else [('AMDF', test_summary_df)])
for algorithm_name, file_results in plot_groups:
    file_results = file_results.reset_index(drop=True)
    names = [name.replace('.wav', '') for name in file_results['file']]
    labels = [f'{name}\nF0={int(n)} / V={int(tp + fn)}'
              for name, n, tp, fn in zip(names, file_results['NumF0'],
                                          file_results['TP'], file_results['FN'])]
    x = np.arange(len(file_results))
    width = 0.35
    f1 = file_results['macro_f1_vu'].to_numpy(dtype=float)
    ba = (file_results['TP'] / (file_results['TP'] + file_results['FN'])
          + file_results['TN'] / (file_results['TN'] + file_results['FP'])).to_numpy(dtype=float) / 2
    mean_error = file_results['abs_error_mean'].to_numpy(dtype=float)
    std_error = file_results['abs_error_std'].to_numpy(dtype=float)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(9.4, 6.0), sharex=True,
                                      layout='constrained')
    for bars in (top.bar(x - width/2, f1, width, label='Macro F1 V/UV', color='#2166ac'),
                 top.bar(x + width/2, ba, width, label='Balanced accuracy', color='#67a9cf')):
        top.bar_label(bars, fmt='%.2f', padding=2, fontsize=8)
    top.set(ylabel='Tỷ lệ', ylim=(0, 1.18), title=f'{algorithm_name}: phân loại V/UV theo file')
    top.legend(loc='upper left', ncol=2, frameon=False)
    top.grid(axis='y', alpha=0.2)
    top.set_axisbelow(True)
    for bars in (bottom.bar(x - width/2, mean_error, width, label='|F0mean − LAB|', color='#b35806'),
                 bottom.bar(x + width/2, std_error, width, label='|F0std − LAB|', color='#f1a340')):
        bottom.bar_label(bars, fmt='%.1f', padding=2, fontsize=8)
    bottom.set(ylabel='Sai số tuyệt đối (Hz)',
               ylim=(0, max(np.nanmax(mean_error), np.nanmax(std_error)) * 1.35),
               title='Thống kê F0 theo file')
    bottom.set_xticks(x, labels)
    bottom.set_xlabel('F0/V là số đếm, không phải recall V')
    bottom.legend(loc='upper right', ncol=2, frameon=False)
    bottom.grid(axis='y', alpha=0.2)
    bottom.set_axisbelow(True)
    fig.suptitle('TEST: mỗi file có một trọng số như nhau trong macro metric')
    plt.show()
    plt.close(fig)
